# Cosmos 3 Nano Reasoner — MMAD zero-shot benchmark (Kaggle T4×2)

This notebook tests the **Cosmos 3 Nano Reasoner-only BNB8 derivative** on a deterministic, label-neutral 140-question MMAD subset (20 questions per capability). It uses official Cosmos 3 Nano Reasoner weights repacked and quantized to 8-bit by the community; it is **not** an official-precision result.

Execution order:
1. Select **GPU T4 ×2** and enable Internet.
2. Run cells top-to-bottom.
3. The notebook first runs a 3-sample smoke gate, then resumes into all 140 questions.
4. Download `/kaggle/working/cosmos3_nano_mmad_140_artifacts.zip` at the end.

Why BNB8: the official unified checkpoint is about 35 GB and its shards mix Reasoner and Generator tensors, which exceeds a normal Kaggle working disk and is not a practical T4 path. The selected checkpoint is about 8.8 GB and excludes Generator tensors. Ground truth is kept out of model prompts and joined only during evaluation. If Hugging Face returns 401, add a private Kaggle secret named `HF_TOKEN`.


In [ ]:
%pip install -q -U "transformers>=5.14.0" accelerate "bitsandbytes>=0.49.0" qwen-vl-utils safetensors remotezip requests pillow pandas matplotlib seaborn


In [ ]:
import os, sys, json, time, shutil, subprocess, platform
from pathlib import Path

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['HF_HOME'] = '/kaggle/working/hf_cache'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

WORK = Path('/kaggle/working')
REPO = WORK / 'mini-world-model'
REPO_URL = 'https://github.com/anhsown/mini-world-model.git'
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO)], check=True)

BASE = REPO / 'research/mmad_model_benchmark'
DATA = WORK / 'mmad_cosmos_subset'
OUT = WORK / 'cosmos3_nano_mmad_140'
CACHE = WORK / 'mmad_archive_cache'
OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(BASE))
print('repo commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
import torch, transformers

assert torch.cuda.is_available(), 'Enable a GPU accelerator before continuing.'
gpu_info = []
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    gpu_info.append({'index': index, 'name': props.name, 'vram_gib': round(props.total_memory / 2**30, 2)})
print(json.dumps({'torch': torch.__version__, 'transformers': transformers.__version__, 'gpus': gpu_info}, indent=2))
assert torch.cuda.device_count() >= 1, 'Select a Kaggle GPU accelerator. T4 x2 is preferred.'
assert min(item['vram_gib'] for item in gpu_info[:2]) >= 14.0, 'Each GPU needs at least 14 GiB.'


In [ ]:
# Deterministic 140-question manifest + robust HTTP-range image download.
subprocess.run([
    sys.executable, str(BASE / 'prepare_subset.py'),
    '--output', str(DATA), '--questions-per-task', '20', '--metadata-only'
], cwd=BASE, check=True)

from prepare_full import materialize_all_images
manifest_path = DATA / 'subset_manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
materialize_all_images(manifest, DATA, CACHE, range_download=True)

missing = [row['image_file'] for row in manifest['records'] if not (DATA / row['image_file']).exists()]
assert len(manifest['records']) == 140 and not missing
assert all(Path(row['image_file']).name.startswith('image_') for row in manifest['records'])
print('manifest:', manifest['manifest_sha256'])
print('questions:', len(manifest['records']), 'unique images:', len({r['image_file'] for r in manifest['records']}))


In [ ]:
# Optional Hugging Face authentication. The token is never printed.
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    hf_token = None
if hf_token:
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
print('HF authentication:', 'private token enabled' if hf_token else 'anonymous/public access')


In [ ]:
# Disk-safe Cosmos 3 Nano Reasoner-only derivative. Generator weights are absent.
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

MODEL_ID = 'ThePyProgrammer/Cosmos3-Nano-reasoner-bnb8-vllm-und-only'
BASE_MODEL_ID = 'nvidia/Cosmos3-Nano'
MAX_NEW_TOKENS = 32
required_free_gib = 11.0
free_gib = shutil.disk_usage(WORK).free / 2**30
print(f'free disk before model download: {free_gib:.2f} GiB')
assert free_gib >= required_free_gib, f'Need at least {required_free_gib} GiB free; found {free_gib:.2f} GiB.'
max_memory = {i: '14GiB' for i in range(torch.cuda.device_count())}
max_memory['cpu'] = '24GiB'
load_started = time.perf_counter()
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=256 * 28 * 28,
    max_pixels=512 * 28 * 28,
    token=hf_token,
)
try:
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        dtype=torch.float16,
        device_map='auto',
        max_memory=max_memory,
        low_cpu_mem_usage=True,
        offload_folder=str(WORK / 'cosmos_offload'),
        offload_state_dict=True,
        attn_implementation='sdpa',
        token=hf_token,
    ).eval()
except Exception as error:
    diagnostic = {'status': 'load_failed', 'error_type': type(error).__name__, 'model': MODEL_ID, 'gpus': gpu_info, 'error': str(error)}
    (OUT / 'load_diagnostic.json').write_text(json.dumps(diagnostic, indent=2), encoding='utf-8')
    raise RuntimeError('Cosmos 3 Nano Reasoner BNB8 did not fit. Preserve load_diagnostic.json for diagnosis.') from error

model.generation_config.do_sample = False
model.generation_config.temperature = None
model.generation_config.top_p = None
model.generation_config.top_k = None
device_map = getattr(model, 'hf_device_map', {})
run_config = {
    'model': MODEL_ID,
    'base_model': BASE_MODEL_ID,
    'precision': 'community BNB8 language layers + FP16 residual/vision, reasoner-only',
    'official_precision_result': False,
    'max_new_tokens': MAX_NEW_TOKENS,
    'manifest_sha256': manifest['manifest_sha256'],
    'load_seconds': round(time.perf_counter() - load_started, 2),
    'gpu_info': gpu_info,
    'device_map': device_map,
}
(OUT / 'run_config.json').write_text(json.dumps(run_config, indent=2, default=str), encoding='utf-8')
print(json.dumps(run_config, indent=2, default=str))
for index in range(torch.cuda.device_count()):
    print(f'cuda:{index} allocated GiB:', round(torch.cuda.memory_allocated(index) / 2**30, 2))


In [ ]:
from datetime import datetime, timezone
from common.mmad import SYSTEM_PROMPT, append_jsonl, evaluate_records, load_jsonl, parse_prediction, write_evaluation
from qwen_vl_utils import process_vision_info

PREDICTIONS = OUT / 'predictions.jsonl'
input_device = model.device

def synchronize_all():
    for index in range(torch.cuda.device_count()):
        torch.cuda.synchronize(index)

def infer_one(sample):
    conversation = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': [
            {'type': 'image', 'image': str((DATA / sample['image_file']).resolve())},
            {'type': 'text', 'text': sample['prompt']},
        ]},
    ]
    text_prompt = processor.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(conversation)
    inputs = processor(
        text=[text_prompt], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors='pt',
    ).to(input_device)
    synchronize_all()
    started = time.perf_counter()
    with torch.inference_mode():
        generated = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    synchronize_all()
    latency = time.perf_counter() - started
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated)]
    raw = processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()
    prediction = parse_prediction(raw)
    return raw, prediction, latency

def run_records(records, label):
    previous = load_jsonl(PREDICTIONS)
    done = {row['sample_id'] for row in previous if row.get('status') in {'ok', 'parse_failure'}}
    pending = [row for row in records if row['sample_id'] not in done]
    print(f'{label}: total={len(records)} done={len(records)-len(pending)} pending={len(pending)}', flush=True)
    started_all = time.perf_counter()
    for index, sample in enumerate(pending, 1):
        try:
            raw, prediction, latency = infer_one(sample)
            status, error = ('ok' if prediction else 'parse_failure'), None
        except torch.OutOfMemoryError as exc:
            torch.cuda.empty_cache()
            raw, prediction, latency, status, error = '', None, 0.0, 'oom', str(exc)
        append_jsonl(PREDICTIONS, {
            'sample_id': sample['sample_id'],
            'model': MODEL_ID,
            'base_model': BASE_MODEL_ID,
            'backend': 'Transformers BNB8 reasoner-only derivative, device_map=auto',
            'manifest_sha256': manifest['manifest_sha256'],
            'status': status,
            'prediction': prediction,
            'raw_response': raw,
            'latency_seconds': round(latency, 4),
            'error': error,
            'created_at': datetime.now(timezone.utc).isoformat(),
        })
        if index % 5 == 0 or index == len(pending):
            elapsed = time.perf_counter() - started_all
            rate = index / max(elapsed, 1e-6)
            eta = (len(pending) - index) / max(rate, 1e-6) / 60
            print(f'[{index}/{len(pending)}] {sample["sample_id"]} status={status} pred={prediction} latency={latency:.2f}s ETA={eta:.1f}m', flush=True)
    return load_jsonl(PREDICTIONS)


In [ ]:
# Smoke gate: three samples from different capabilities.
smoke_indices = [0, 20, 120]
smoke_records = [manifest['records'][index] for index in smoke_indices]
predictions = run_records(smoke_records, 'SMOKE')
smoke_ids = {row['sample_id'] for row in smoke_records}
smoke_outputs = [row for row in predictions if row['sample_id'] in smoke_ids]
print(json.dumps(smoke_outputs, ensure_ascii=False, indent=2))
assert any(row.get('status') == 'ok' for row in smoke_outputs), 'Smoke gate produced no parseable answer; inspect raw responses before continuing.'


In [ ]:
# Continue/resume all 140 balanced questions.
predictions = run_records(manifest['records'], 'MMAD-140')
summary, scored = evaluate_records(manifest, predictions)
write_evaluation(OUT, summary, scored)
print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
# Visual analysis.
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

scored_df = pd.DataFrame(scored)
task_scores = scored_df.groupby('question_type')['correct'].mean().sort_values()
source_scores = scored_df.groupby('source_dataset')['correct'].mean().sort_values()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
task_scores.mul(100).plot.barh(ax=axes[0], color='#76b900', title='Accuracy by MMAD capability')
axes[0].set_xlabel('Accuracy (%)'); axes[0].set_xlim(0, 100)
source_scores.mul(100).plot.barh(ax=axes[1], color='#4c78a8', title='Accuracy by source dataset')
axes[1].set_xlabel('Accuracy (%)'); axes[1].set_xlim(0, 100)
plt.tight_layout()
figure_path = OUT / 'cosmos3_mmad_accuracy.png'
plt.savefig(figure_path, dpi=160, bbox_inches='tight')
plt.show()

display(scored_df[['sample_id', 'question_type', 'source_dataset', 'truth', 'prediction', 'correct', 'latency_seconds']].head(20))


In [ ]:
# Portable artifact; no model weights are included.
archive_base = WORK / 'cosmos3_nano_mmad_140_artifacts'
archive_path = shutil.make_archive(str(archive_base), 'zip', OUT)
print('DOWNLOAD:', archive_path)
print('size MiB:', round(Path(archive_path).stat().st_size / 2**20, 2))
